## Libraries and Packages

In [1]:
import os
!pip install textstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.1/239.1 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.7/939.7 kB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 56.0 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Preprocessing

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from pathlib import Path

In [ ]:
train_df = pd.read_excel('/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/Generated_Datasets/4_Binary_Final/binary_train_file.xlsx')
eval_df = pd.read_excel('/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/Generated_Datasets/4_Binary_Final/binary_valid_file.xlsx')
test_df = pd.read_excel('/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/Generated_Datasets/4_Binary_Final/binary_test_file.xlsx')

In [ ]:
train_df.head()

,text,author,label
0,However the hardware simulated by your virtual...,1,0
1,Note: Looks like VirtualBox's website is tempo...,2,0
2,Modern CI/CD pipelines leverage infrastructure...,5,1
3,"Generative AI models, like large language mode...",1,0
4,"RAID configurations, such as RAID 10, provide ...",2,0


### Total Samples Distribution

In [ ]:
# Function to count rows per author for a given DataFrame
def count_rows_per_author(df, dataset_name):
    author_counts = df['author'].value_counts().reset_index()
    author_counts.columns = ['author', 'count']
    author_counts['dataset'] = dataset_name  # Add a column to identify the dataset
    return author_counts

# Get row counts for each DataFrame
train_counts = count_rows_per_author(train_df, "Train")
eval_counts = count_rows_per_author(eval_df, "Validation")
test_counts = count_rows_per_author(test_df, "Test")

# Combine all counts into a single DataFrame
combined_counts = pd.concat([train_counts, eval_counts, test_counts])

# Create a grouped bar chart
fig = px.bar(combined_counts, x='author', y='count', color='dataset', barmode='group',
             labels={'author': 'Author', 'count': 'Number of Rows'},
             title="Total Number of Rows per Author Across All DataFrames",
             text='count')

# Format and position the text
fig.update_traces(texttemplate='%{text}', textposition='outside')

# Update layout for better readability
fig.update_layout(
    xaxis_title='Author',
    yaxis_title='Number of Rows',
    legend_title='Dataset'
)

# Show the plot
fig.show()

### Dublicate Rows, Authors & DF

In [ ]:
# Function to find duplicated text for each author and count them
def find_duplicated_text_counts(df):
    # Group by 'author' and 'text', then count occurrences
    grouped = df.groupby(['author', 'text']).size().reset_index(name='count')

    # Filter rows where count > 1 (duplicated text for the author)
    duplicated_texts = grouped[grouped['count'] > 1]

    # Group by 'author' and count the number of duplicated texts
    duplicated_counts = duplicated_texts.groupby('author').size().reset_index(name='duplicated_count')

    return duplicated_counts

# Find duplicated text counts for each DataFrame
train_duplicates = find_duplicated_text_counts(train_df)
eval_duplicates = find_duplicated_text_counts(eval_df)
test_duplicates = find_duplicated_text_counts(test_df)

# Add a 'dataset' column to identify the source DataFrame
train_duplicates['dataset'] = 'train'
eval_duplicates['dataset'] = 'eval'
test_duplicates['dataset'] = 'test'

# Combine the results into a single DataFrame
combined_duplicates = pd.concat([train_duplicates, eval_duplicates, test_duplicates])

# Get the list of all unique authors across all datasets
all_authors = pd.concat([train_df, eval_df, test_df])['author'].unique()

# Ensure all authors are included in the combined_duplicates DataFrame
# Create a DataFrame with all authors and datasets, then merge with combined_duplicates
all_authors_df = pd.DataFrame({
    'author': all_authors
})

# Create a cartesian product of all authors and datasets
all_combinations = pd.MultiIndex.from_product(
    [all_authors, ['train', 'eval', 'test']],
    names=['author', 'dataset']
).to_frame(index=False)

# Merge with combined_duplicates to fill missing values with 0
combined_duplicates = all_combinations.merge(
    combined_duplicates,
    on=['author', 'dataset'],
    how='left'
).fillna({'duplicated_count': 0})

# Plot the count of duplicated texts using Plotly
fig = px.bar(combined_duplicates, x='author', y='duplicated_count', color='dataset', barmode='group',
             text='duplicated_count', title='Count of Duplicated Texts by Author and Dataset')

# Show the count outside the bars
fig.update_traces(texttemplate='%{text}', textposition='outside')

# Update layout for better readability
fig.update_layout(
    xaxis_title='Author',
    yaxis_title='Count of Duplicated Texts',
    legend_title='Dataset'
)

# Show the plot
fig.show()

In [ ]:
# Function to find and display two duplicate samples for each author
def find_and_display_duplicates(df, dataset_name):
    print(f"\nDuplicate samples for {dataset_name} dataset:")

    # Group by 'author' and 'text' to find duplicates
    grouped = df.groupby(['author', 'text']).size().reset_index(name='count')

    # Filter rows where count > 1 (duplicates)
    duplicates = grouped[grouped['count'] > 1]

    # Iterate over each author and display two duplicate samples
    for author in df['author'].unique():
        author_duplicates = duplicates[duplicates['author'] == author]

        if not author_duplicates.empty:
            print(f"\nAuthor: {author}")
            for _, row in author_duplicates.head(2).iterrows():  # Limit to 2 duplicates
                duplicate_text = row['text']
                duplicate_indices = df[(df['author'] == author) & (df['text'] == duplicate_text)].index.tolist()
                print(f"Duplicate text: '{duplicate_text}'")
                print(f"Indices: {duplicate_indices}")
        else:
            print(f"\nAuthor: {author} has no duplicates.")

# Find and display duplicates for each DataFrame
find_and_display_duplicates(train_df, "train")
find_and_display_duplicates(eval_df, "eval")
find_and_display_duplicates(test_df, "test")


Duplicate samples for train dataset:

Author: 1
Duplicate text: '0'
Indices: [38495, 45969]
Duplicate text: 'Containerization technologies like Docker and Kubernetes simplify application deployment and management by packaging applications and their dependencies into isolated containers.  This improves portability and scalability across various environments.'
Indices: [30067, 52320]

Author: 2
Duplicate text: 'Containerization technologies like Docker and Kubernetes simplify application deployment and management across different environments.  Their lightweight nature and efficient resource utilization make them ideal for microservices architectures and cloud-native applications.'
Indices: [22849, 25005, 44809]
Duplicate text: 'Containerization technologies like Docker and Kubernetes simplify application deployment and management across various environments.  Their lightweight nature and efficient resource utilization contribute to cost savings and improved scalability in cloud-based d

### Drop Dublicates

In [ ]:
# Function to delete duplicates while keeping the first occurrence
def delete_duplicates(df):
    # Drop duplicates based on 'author' and 'text', keeping the first occurrence
    df = df.drop_duplicates(subset=['author', 'text'], keep='first')
    return df

# Delete duplicates in all DataFrames
train_df = delete_duplicates(train_df)
eval_df = delete_duplicates(eval_df)
test_df = delete_duplicates(test_df)

In [ ]:
# Function to find duplicated text for each author and count them
def find_duplicated_text_counts(df):
    # Group by 'author' and 'text', then count occurrences
    grouped = df.groupby(['author', 'text']).size().reset_index(name='count')

    # Filter rows where count > 1 (duplicated text for the author)
    duplicated_texts = grouped[grouped['count'] > 1]

    # Group by 'author' and count the number of duplicated texts
    duplicated_counts = duplicated_texts.groupby('author').size().reset_index(name='duplicated_count')

    return duplicated_counts

# Find duplicated text counts for each DataFrame
train_duplicates = find_duplicated_text_counts(train_df)
eval_duplicates = find_duplicated_text_counts(eval_df)
test_duplicates = find_duplicated_text_counts(test_df)

# Add a 'dataset' column to identify the source DataFrame
train_duplicates['dataset'] = 'train'
eval_duplicates['dataset'] = 'eval'
test_duplicates['dataset'] = 'test'

# Combine the results into a single DataFrame
combined_duplicates = pd.concat([train_duplicates, eval_duplicates, test_duplicates])

# Get the list of all unique authors across all datasets
all_authors = pd.concat([train_df, eval_df, test_df])['author'].unique()

# Ensure all authors are included in the combined_duplicates DataFrame
# Create a DataFrame with all authors and datasets, then merge with combined_duplicates
all_authors_df = pd.DataFrame({
    'author': all_authors
})

# Create a cartesian product of all authors and datasets
all_combinations = pd.MultiIndex.from_product(
    [all_authors, ['train', 'eval', 'test']],
    names=['author', 'dataset']
).to_frame(index=False)

# Merge with combined_duplicates to fill missing values with 0
combined_duplicates = all_combinations.merge(
    combined_duplicates,
    on=['author', 'dataset'],
    how='left'
).fillna({'duplicated_count': 0})

# Plot the count of duplicated texts using Plotly
fig = px.bar(combined_duplicates, x='author', y='duplicated_count', color='dataset', barmode='group',
             text='duplicated_count', title='Count of Duplicated Texts by Author and Dataset')

# Show the count outside the bars
fig.update_traces(texttemplate='%{text}', textposition='outside')

# Update layout for better readability
fig.update_layout(
    xaxis_title='Author',
    yaxis_title='Count of Duplicated Texts',
    legend_title='Dataset'
)

# Show the plot
fig.show()

### Drop Author

In [ ]:
if 'author' in train_df.columns:
  train_df.drop(columns=['author'], inplace=True)
if 'author' in eval_df.columns:
  eval_df.drop(columns=['author'], inplace=True)
if 'author' in test_df.columns:
  test_df.drop(columns=['author'], inplace=True)

train_df.head()

,text,label
0,However the hardware simulated by your virtual...,0
1,Note: Looks like VirtualBox's website is tempo...,0
2,Modern CI/CD pipelines leverage infrastructure...,1
3,"Generative AI models, like large language mode...",0
4,"RAID configurations, such as RAID 10, provide ...",0


### Nulls or NaN etc

In [ ]:
# Define a list of garbage values (customize as needed)
garbage_values = ['N/A', 'null', 'NULL', 'NaN', 'nan', 'None', 'none', '']

# Function to find nulls, NaNs, and garbage values in the 'text' column
def find_garbage_values(df, dataset_name):
    print(f"\n=======================Checking for garbage values in {dataset_name} dataset ==========================")

    # Check for nulls or NaNs
    null_mask = df['text'].isna()
    null_rows = df[null_mask]
    print(f"Null or NaN values:\n{null_rows}\n")

    # Check for empty strings or whitespace-only strings
    empty_mask = df['text'].str.strip().eq('') | df['text'].isna()
    empty_rows = df[empty_mask]
    print(f"Empty or whitespace-only strings:\n{empty_rows}\n")

    # Check for other garbage values
    garbage_mask = df['text'].isin(garbage_values)
    garbage_rows = df[garbage_mask]
    print(f"Other garbage values:\n{garbage_rows}\n")

    # Combine all masks to get rows with any garbage value
    combined_mask = null_mask | empty_mask | garbage_mask
    combined_rows = df[combined_mask]
    print(f"All rows with garbage values:\n{combined_rows}\n")

    return combined_rows

# Find garbage values in each DataFrame
train_garbage = find_garbage_values(train_df, "Train")
eval_garbage = find_garbage_values(eval_df, "Validation")
test_garbage = find_garbage_values(test_df, "Test")


=======================Checking for garbage values in Train dataset ==========================
Null or NaN values:
Empty DataFrame
Columns: [text, label]
Index: []

Empty or whitespace-only strings:
Empty DataFrame
Columns: [text, label]
Index: []

Other garbage values:
Empty DataFrame
Columns: [text, label]
Index: []

All rows with garbage values:
Empty DataFrame
Columns: [text, label]
Index: []


=======================Checking for garbage values in Validation dataset ==========================
Null or NaN values:
Empty DataFrame
Columns: [text, label]
Index: []

Empty or whitespace-only strings:
Empty DataFrame
Columns: [text, label]
Index: []

Other garbage values:
Empty DataFrame
Columns: [text, label]
Index: []

All rows with garbage values:
Empty DataFrame
Columns: [text, label]
Index: []


=======================Checking for garbage values in Test dataset ==========================
Null or NaN values:
Empty DataFrame
Columns: [text, label]
Index: []

Empty or whitespace-only s

In [ ]:
# Define a list of garbage values (customize as needed)
garbage_values = ['N/A', 'null', 'NULL', 'NaN', 'nan', 'None', 'none', '']

# Function to find nulls, NaNs, and garbage values in the 'text' column
def find_garbage_values(df):
    # Check for nulls or NaNs
    null_mask = df['text'].isna()

    # Check for empty strings or whitespace-only strings
    empty_mask = df['text'].str.strip().eq('') | df['text'].isna()

    # Check for other garbage values
    garbage_mask = df['text'].isin(garbage_values)

    # Combine all masks to get rows with any garbage value
    combined_mask = null_mask | empty_mask | garbage_mask
    combined_rows = df[combined_mask]

    return combined_rows

# Find garbage rows in each DataFrame
train_garbage = find_garbage_values(train_df)
eval_garbage = find_garbage_values(eval_df)
test_garbage = find_garbage_values(test_df)

# Count the number of garbage rows in each DataFrame
garbage_counts = {
    'train': len(train_garbage),
    'eval': len(eval_garbage),
    'test': len(test_garbage)
}

# Convert the counts to a DataFrame for plotting
garbage_counts_df = pd.DataFrame({
    'dataset': list(garbage_counts.keys()),
    'garbage_count': list(garbage_counts.values())
})

# Plot the count of garbage rows using Plotly
fig = px.bar(garbage_counts_df, x='dataset', y='garbage_count',
             labels={'dataset': 'Dataset', 'garbage_count': 'Count of Garbage Rows'},
             title='Count of Garbage Rows in Each Dataset',
             text='garbage_count')

# Show the count outside the bars
fig.update_traces(texttemplate='%{text}', textposition='outside')

# Update layout for better readability
fig.update_layout(
    xaxis_title='Dataset',
    yaxis_title='Count of Garbage Rows',
    showlegend=False
)

# Show the plot
fig.show()

In [ ]:
garbage_values = ['N/A', 'null', 'NULL', 'NaN', 'nan', 'None', 'none', '']

def drop_garbage_values(df):
    null_mask = df['text'].isna()
    empty_mask = df['text'].str.strip().eq('') | df['text'].isna()
    garbage_mask = df['text'].isin(garbage_values)
    combined_mask = null_mask | empty_mask | garbage_mask
    cleaned_df = df[~combined_mask]

    return cleaned_df

train_df = drop_garbage_values(train_df)
eval_df = drop_garbage_values(eval_df)
test_df = drop_garbage_values(test_df)

In [ ]:
# Define a list of garbage values (customize as needed)
garbage_values = ['N/A', 'null', 'NULL', 'NaN', 'nan', 'None', 'none', '']

# Function to find nulls, NaNs, and garbage values in the 'text' column
def find_garbage_values(df):
    # Check for nulls or NaNs
    null_mask = df['text'].isna()

    # Check for empty strings or whitespace-only strings
    empty_mask = df['text'].str.strip().eq('') | df['text'].isna()

    # Check for other garbage values
    garbage_mask = df['text'].isin(garbage_values)

    # Combine all masks to get rows with any garbage value
    combined_mask = null_mask | empty_mask | garbage_mask
    combined_rows = df[combined_mask]

    return combined_rows

# Find garbage rows in each DataFrame
train_garbage = find_garbage_values(train_df)
eval_garbage = find_garbage_values(eval_df)
test_garbage = find_garbage_values(test_df)

# Count the number of garbage rows in each DataFrame
garbage_counts = {
    'train': len(train_garbage),
    'eval': len(eval_garbage),
    'test': len(test_garbage)
}

# Convert the counts to a DataFrame for plotting
garbage_counts_df = pd.DataFrame({
    'dataset': list(garbage_counts.keys()),
    'garbage_count': list(garbage_counts.values())
})

# Plot the count of garbage rows using Plotly
fig = px.bar(garbage_counts_df, x='dataset', y='garbage_count',
             labels={'dataset': 'Dataset', 'garbage_count': 'Count of Garbage Rows'},
             title='Count of Garbage Rows in Each Dataset',
             text='garbage_count')

# Show the count outside the bars
fig.update_traces(texttemplate='%{text}', textposition='outside')

# Update layout for better readability
fig.update_layout(
    xaxis_title='Dataset',
    yaxis_title='Count of Garbage Rows',
    showlegend=False
)

# Show the plot
fig.show()

### Check Data Leakage

In [ ]:
train_texts = set(train_df['text'])
valid_texts = set(eval_df['text'])
test_texts = set(test_df['text'])

# Check for common samples
common_train_valid = train_texts.intersection(valid_texts)
common_train_test = train_texts.intersection(test_texts)
common_valid_test = valid_texts.intersection(test_texts)

print(f"🔍 Common samples between Train & Validation: {len(common_train_valid)}")
print(f"🔍 Common samples between Train & Test: {len(common_train_test)}")
print(f"🔍 Common samples between Validation & Test: {len(common_valid_test)}")

🔍 Common samples between Train & Validation: 176
🔍 Common samples between Train & Test: 162
🔍 Common samples between Validation & Test: 77


In [ ]:
eval_df = eval_df[~eval_df["text"].isin(train_texts)]
test_df = test_df[~test_df["text"].isin(train_texts)]
test_df = test_df[~test_df["text"].isin(valid_texts)]

### Save Dataset

In [ ]:
train_df.to_excel('/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/Generated_Datasets/4_Binary_Final/binary_train_file_refined.xlsx', index=False)
eval_df.to_excel('/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/Generated_Datasets/4_Binary_Final/binary_valid_file_refined.xlsx', index=False)
test_df.to_excel('/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/Generated_Datasets/4_Binary_Final/binary_test_file_refined.xlsx', index=False)

# Data Generation for Author Change Detection

In [ ]:
import pandas as pd

# Read the datasets
train_df = pd.read_excel('/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/Generated_Datasets/5_Binary_Single_vs_Multiauth/train.xlsx')
eval_df = pd.read_excel('/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/Generated_Datasets/5_Binary_Single_vs_Multiauth/valid.xlsx')
test_df = pd.read_excel('/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/Generated_Datasets/5_Binary_Single_vs_Multiauth/test.xlsx')

In [ ]:
import pandas as pd

# Ensure text and changes columns are valid
for df in [train_df, eval_df, test_df]:
    df["text"] = df["text"].astype(str).fillna("")
    df["changes"] = df["changes"].apply(lambda x: eval(x) if isinstance(x, str) else x)

# Correct paragraph splitter using real newline
def split_into_paragraphs(text):
    return [p.strip() for p in text.split("\n") if p.strip()]

# Create (text1, text2) pairs with corresponding change labels
def create_pairs_and_labels(text, changes):
    paragraphs = split_into_paragraphs(text)
    pairs = []
    labels = []
    for i in range(len(paragraphs) - 1):
        pairs.append((paragraphs[i], paragraphs[i + 1]))
        try:
            labels.append(changes[i])
        except IndexError:
            # In case changes are one short or corrupted
            labels.append(0)  # Default label or handle gracefully
    return pairs, labels

# Expand the dataset by paragraph pairs
def expand_df(df, split_name=""):
    all_pairs = []
    all_labels = []
    skipped_rows = 0

    for idx, row in df.iterrows():
        try:
            text = row["text"]
            changes = row["changes"]
            if isinstance(changes, str):
                changes = eval(changes)

            pairs, labels = create_pairs_and_labels(text, changes)

            all_pairs.extend(pairs)
            all_labels.extend(labels)

        except Exception as e:
            print(f"[{split_name}] Skipping row {idx} due to error: {e}")
            skipped_rows += 1

    result_df = pd.DataFrame(all_pairs, columns=["text1", "text2"])

    if len(all_labels) == len(result_df):
        result_df["label"] = all_labels
    else:
        print(f"[{split_name}] Mismatch between pairs and labels: {len(all_pairs)} vs {len(all_labels)}")

    print(f"[{split_name}] Final size: {result_df.shape}, Skipped rows: {skipped_rows}")
    return result_df

# Apply transformation
new_train_df = expand_df(train_df, "Train")
new_eval_df = expand_df(eval_df, "Eval")
new_test_df = expand_df(test_df, "Test")

# Check results
print("Train columns:", new_train_df.columns.tolist())
print("Training Data Distribution:")
print(new_train_df["label"].value_counts())

print("Validation Data Distribution:")
print(new_eval_df["label"].value_counts())

print("Test Data Distribution:")
print(new_test_df["label"].value_counts())

[Train] Final size: (74286, 3), Skipped rows: 0
[Eval] Final size: (16799, 3), Skipped rows: 0
[Test] Final size: (16692, 3), Skipped rows: 0
Train columns: ['text1', 'text2', 'label']
Training Data Distribution:
label
1    51135
0    23151
Name: count, dtype: int64
Validation Data Distribution:
label
1    10920
0     5879
Name: count, dtype: int64
Test Data Distribution:
label
1    10883
0     5809
Name: count, dtype: int64


In [ ]:
new_train_df

,text1,text2,label
0,As stephelton said in the comments to your que...,"Efficient RAID configurations, like RAID 10, o...",1
1,"Efficient RAID configurations, like RAID 10, o...","Containerization technologies, such as Docker,...",1
2,"Containerization technologies, such as Docker,...",Debugging complex software often requires util...,0
3,Debugging complex software often requires util...,The increasing adoption of serverless architec...,0
4,The increasing adoption of serverless architec...,Modern GPUs accelerate machine learning model ...,0
...,...,...,...
74281,both containing Japanese text. If you copy som...,the answer is when you copy some text it get c...,1
74282,the answer is when you copy some text it get c...,"Yes, you need to use an editor that supports m...",1
74283,"Yes, you need to use an editor that supports m...","For plain text files, in general you cannot te...",1
74284,"For plain text files, in general you cannot te...",But it is very much possible that you can use ...,0


In [ ]:
# Save files
output_dir = "/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/Generated_Datasets/6_Change_Recognition/"
new_train_df.to_excel(f"{output_dir}train.xlsx", index=False)
new_eval_df.to_excel(f"{output_dir}valid.xlsx", index=False)
new_test_df.to_excel(f"{output_dir}test.xlsx", index=False)

# Model

In [3]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from sklearn.utils import resample, class_weight
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import warnings
warnings.filterwarnings("ignore")

In [4]:
train_df = pd.read_excel('/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/Generated_Datasets/6_Change_Recognition/train.xlsx')
eval_df = pd.read_excel('/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/Generated_Datasets/6_Change_Recognition/valid.xlsx')
test_df = pd.read_excel('/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/Generated_Datasets/6_Change_Recognition/test.xlsx')

print(train_df.head())

                                               text1  \
0  As stephelton said in the comments to your que...   
1  Efficient RAID configurations, like RAID 10, o...   
2  Containerization technologies, such as Docker,...   
3  Debugging complex software often requires util...   
4  The increasing adoption of serverless architec...   

                                               text2  label  
0  Efficient RAID configurations, like RAID 10, o...      1  
1  Containerization technologies, such as Docker,...      1  
2  Debugging complex software often requires util...      0  
3  The increasing adoption of serverless architec...      0  
4  Modern GPUs accelerate machine learning model ...      0  


In [5]:
print("Training Data Distribution:")
print(train_df["label"].value_counts())

print("Validation Data Distribution:")
print(eval_df["label"].value_counts())

print("Test Data Distribution:")
print(test_df["label"].value_counts())

Training Data Distribution:
label
1    51135
0    23151
Name: count, dtype: int64
Validation Data Distribution:
label
1    10920
0     5879
Name: count, dtype: int64
Test Data Distribution:
label
1    10883
0     5809
Name: count, dtype: int64


### Classes Analysis

In [6]:
print(train_df.dtypes)  # Check data types of columns
print(train_df["text1"].apply(type).value_counts())  # Check for unexpected types
print(train_df["text1"].isnull().sum())  # Check for missing values
print(train_df["text2"].apply(type).value_counts())  # Check for unexpected types
print(train_df["text2"].isnull().sum())  # Check for missing values

print(eval_df.dtypes)  # Check data types of columns
print(eval_df["text1"].apply(type).value_counts())  # Check for unexpected types
print(eval_df["text1"].isnull().sum())  # Check for missing values
print(eval_df["text2"].apply(type).value_counts())  # Check for unexpected types
print(eval_df["text2"].isnull().sum())  # Check for missing values

print(test_df.dtypes)  # Check data types of columns
print(test_df["text1"].apply(type).value_counts())  # Check for unexpected types
print(test_df["text1"].isnull().sum())  # Check for missing values
print(test_df["text2"].apply(type).value_counts())  # Check for unexpected types
print(test_df["text2"].isnull().sum())  # Check for missing values

text1    object
text2    object
label     int64
dtype: object
text1
<class 'str'>      74279
<class 'float'>        7
Name: count, dtype: int64
7
text2
<class 'str'>      74279
<class 'float'>        7
Name: count, dtype: int64
7
text1    object
text2    object
label     int64
dtype: object
text1
<class 'str'>      16798
<class 'float'>        1
Name: count, dtype: int64
1
text2
<class 'str'>      16798
<class 'float'>        1
Name: count, dtype: int64
1
text1    object
text2    object
label     int64
dtype: object
text1
<class 'str'>    16692
Name: count, dtype: int64
0
text2
<class 'str'>    16692
Name: count, dtype: int64
0


In [7]:
train_df["text1"] = train_df["text1"].astype(str).fillna("")
train_df["text2"] = train_df["text2"].astype(str).fillna("")
eval_df["text1"] = eval_df["text1"].astype(str).fillna("")
eval_df["text2"] = eval_df["text2"].astype(str).fillna("")
test_df["text1"] = test_df["text1"].astype(str).fillna("")
test_df["text2"] = test_df["text2"].astype(str).fillna("")

### Sentence Analysis

In [ ]:
# import pandas as pd
# import numpy as np
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.decomposition import LatentDirichletAllocation
# from textstat import flesch_reading_ease, dale_chall_readability_score
# from collections import Counter
# import seaborn as sns
# import matplotlib.pyplot as plt

# def extract_top_keywords(df, top_n=10):
#     """Extracts top N keywords for each class using TF-IDF."""
#     vectorizer = TfidfVectorizer(max_features=5000, stop_words=None)
#     X_tfidf = vectorizer.fit_transform(df["text"])
#     feature_names = vectorizer.get_feature_names_out()

#     top_keywords = {}
#     for label in df["label"].unique():
#         class_indices = df[df["label"] == label].index
#         class_tfidf = X_tfidf[class_indices].mean(axis=0).A1
#         top_words = [feature_names[i] for i in class_tfidf.argsort()[-top_n:]]
#         top_keywords[label] = top_words

#     return top_keywords

# def extract_topics(df, num_topics=5, num_words=10):
#     """Performs topic modeling using LDA for each class."""
#     vectorizer = TfidfVectorizer(max_features=5000, stop_words=None)
#     X_tfidf = vectorizer.fit_transform(df["text"])
#     feature_names = vectorizer.get_feature_names_out()

#     lda = LatentDirichletAllocation(n_components=num_topics, random_state=42)
#     lda.fit(X_tfidf)

#     topic_words = {}
#     for topic_idx, topic in enumerate(lda.components_):
#         words = [feature_names[i] for i in topic.argsort()[-num_words:]]
#         topic_words[f"Topic {topic_idx+1}"] = words

#     return topic_words

# def extract_linguistic_features(df):
#     """Extracts basic linguistic features to compare both classes."""
#     results = []
#     for label in df["label"].unique():
#         class_texts = df[df["label"] == label]["text"]

#         avg_sentence_length = np.mean([len(text.split()) for text in class_texts])
#         avg_flesch_score = np.mean([flesch_reading_ease(text) for text in class_texts])
#         avg_dale_score = np.mean([dale_chall_readability_score(text) for text in class_texts])
#         unique_word_ratio = np.mean([len(set(text.split())) / len(text.split()) for text in class_texts])

#         results.append({
#             "Label": label,
#             "Avg Sentence Length": avg_sentence_length,
#             "Flesch Reading Ease": avg_flesch_score,
#             "Dale-Chall Score": avg_dale_score,
#             "Unique Word Ratio": unique_word_ratio,
#         })

#     return pd.DataFrame(results)

# def plot_top_keywords(top_keywords):
#     """Plots the top keywords for both classes."""
#     for label, words in top_keywords.items():
#         plt.figure(figsize=(10, 5))
#         word_counts = Counter(words)
#         sns.barplot(x=list(word_counts.values()), y=list(word_counts.keys()))
#         plt.title(f"Top Keywords for Class {label}")
#         plt.xlabel("TF-IDF Importance")
#         plt.ylabel("Words")
#         plt.show()

# # Run the analysis
# print("Extracting Top Keywords...")
# top_keywords = extract_top_keywords(train_df)
# print(top_keywords)

# print("\nExtracting Topics...")
# topics = extract_topics(train_df)
# print(topics)

# print("\nExtracting Linguistic Features...")
# linguistic_features = extract_linguistic_features(train_df)
# print(linguistic_features)

# # Plot top keywords
# plot_top_keywords(top_keywords)

In [ ]:
# import pandas as pd
# import numpy as np
# import re
# from collections import Counter
# import seaborn as sns
# import matplotlib.pyplot as plt
# import spacy
# from sklearn.feature_extraction.text import CountVectorizer
# from textstat import flesch_reading_ease, dale_chall_readability_score
# from transformers import GPT2Tokenizer, GPT2LMHeadModel
# import torch

# # Load SpaCy for POS tagging and dependency parsing
# nlp = spacy.load("en_core_web_sm")

# # Load GPT-2 for Perplexity scoring
# tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
# model = GPT2LMHeadModel.from_pretrained("gpt2")

# def calculate_perplexity(sentence):
#     """Computes the perplexity of a sentence using GPT-2."""
#     encodings = tokenizer(sentence, return_tensors="pt")
#     with torch.no_grad():
#         outputs = model(**encodings, labels=encodings["input_ids"])
#     loss = outputs.loss
#     return np.exp(loss.item())

# def extract_ngrams(texts, n=2, top_k=10):
#     """Finds the most common n-grams (bigram/trigram)."""
#     vectorizer = CountVectorizer(ngram_range=(n, n), stop_words="english")
#     X = vectorizer.fit_transform(texts)
#     freqs = zip(vectorizer.get_feature_names_out(), X.toarray().sum(axis=0))
#     return sorted(freqs, key=lambda x: -x[1])[:top_k]

# def analyze_readability(texts):
#     """Computes readability metrics."""
#     return {
#         "Flesch Reading Ease": np.mean([flesch_reading_ease(t) for t in texts]),
#         "Dale-Chall Score": np.mean([dale_chall_readability_score(t) for t in texts]),
#     }

# def analyze_pos_distribution(texts):
#     """Computes part-of-speech (POS) distribution."""
#     pos_counts = Counter()
#     total_words = 0

#     for text in texts:
#         doc = nlp(text)
#         for token in doc:
#             pos_counts[token.pos_] += 1
#             total_words += 1

#     return {pos: count / total_words for pos, count in pos_counts.items()}

# def analyze_punctuation(texts):
#     """Counts different punctuation usage patterns."""
#     punct_counts = Counter()
#     total_chars = sum(len(t) for t in texts)

#     for text in texts:
#         punct_counts["commas"] += text.count(",")
#         punct_counts["periods"] += text.count(".")
#         punct_counts["question_marks"] += text.count("?")
#         punct_counts["exclamation_marks"] += text.count("!")
#         punct_counts["quotation_marks"] += text.count('"')
#         punct_counts["colons_semicolons"] += text.count(":") + text.count(";")

#     return {p: count / total_chars for p, count in punct_counts.items()}

# def analyze_sentence_structure(texts):
#     """Analyzes sentence length variability and dependency parsing."""
#     sentence_lengths = []
#     dependency_patterns = Counter()

#     for text in texts:
#         doc = nlp(text)
#         for sent in doc.sents:
#             sentence_lengths.append(len(sent))
#             for token in sent:
#                 dependency_patterns[token.dep_] += 1

#     avg_sentence_length = np.mean(sentence_lengths)
#     std_sentence_length = np.std(sentence_lengths)
#     return avg_sentence_length, std_sentence_length, dependency_patterns.most_common(5)

# def perform_analysis(df):
#     """Runs all analysis methods and prints results for both classes."""
#     for label in [0, 1]:
#         print(f"\n##### Analysis for Class {label} #####")

#         texts = df[df["label"] == label]["text"].tolist()

#         # N-Gram Analysis
#         print(f"Top 10 Bigrams: {extract_ngrams(texts, n=2)}")
#         print(f"Top 10 Trigrams: {extract_ngrams(texts, n=3)}")

#         # Readability Metrics
#         readability_scores = analyze_readability(texts)
#         print(f"Readability Metrics: {readability_scores}")

#         # POS Distribution
#         pos_distribution = analyze_pos_distribution(texts)
#         print(f"POS Distribution: {pos_distribution}")

#         # Punctuation Analysis
#         punctuation_usage = analyze_punctuation(texts)
#         print(f"Punctuation Usage: {punctuation_usage}")

#         # Sentence Structure
#         avg_len, std_len, top_dependencies = analyze_sentence_structure(texts)
#         print(f"Sentence Length: Avg = {avg_len}, Std Dev = {std_len}")
#         print(f"Most Common Dependency Structures: {top_dependencies}")

#         # Perplexity (LLM Detector)
#         perplexities = [calculate_perplexity(sent) for sent in texts[:50]]  # Sample 50 sentences
#         print(f"Average Perplexity: {np.mean(perplexities)}")

# # Run the analysis on your balanced dataset
# perform_analysis(train_df)


### Data Balancing Techniques Comparison

In [ ]:
# import pandas as pd
# import numpy as np
# from sklearn.utils import resample
# from imblearn.over_sampling import SMOTE, ADASYN
# from imblearn.under_sampling import RandomUnderSampler, NearMiss, TomekLinks
# from imblearn.combine import SMOTEENN, SMOTETomek
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.preprocessing import LabelEncoder
# from sklearn.decomposition import PCA
# from sentence_transformers import SentenceTransformer
# from tqdm import tqdm
# from scipy.spatial.distance import cosine
# from imblearn.over_sampling import RandomOverSampler

# # Load BERT Model once (global to avoid reloading)
# bert_model = SentenceTransformer("all-MiniLM-L6-v2")

# def compute_bert_embeddings(texts, batch_size=256):
#     """Compute BERT embeddings efficiently using batching."""
#     return np.array(bert_model.encode(texts, batch_size=batch_size, show_progress_bar=True))

# def preprocess_features(df, max_features=500, pca_components=100):
#     """Precompute TF-IDF, BERT embeddings, and numeric features."""
#     if "label" not in df.columns or "text" not in df.columns:
#         raise ValueError("Dataset must contain 'text' and 'label' columns.")

#     numeric_columns = df.select_dtypes(exclude=["object"]).columns.drop("label", errors="ignore").tolist()

#     # TF-IDF Vectorization
#     tfidf = TfidfVectorizer(max_features=max_features)
#     tfidf_features = pd.DataFrame(tfidf.fit_transform(df["text"].astype(str)).toarray(),
#                                   columns=[f"tfidf_{i}" for i in range(max_features)])

#     # BERT Embeddings with batching
#     bert_embeddings = compute_bert_embeddings(df["text"].astype(str).tolist())

#     # Reduce dimensionality of BERT embeddings using PCA
#     pca = PCA(n_components=pca_components)
#     bert_embeddings_pca = pca.fit_transform(bert_embeddings)
#     df_bert = pd.DataFrame(bert_embeddings_pca, columns=[f"bert_pca_{i}" for i in range(pca_components)])

#     # Encode categorical columns
#     for col in numeric_columns:
#         if df[col].dtype == "object":
#             df[col] = LabelEncoder().fit_transform(df[col])

#     # Combine all features
#     X = pd.concat([df[numeric_columns], tfidf_features, df_bert], axis=1).copy()
#     y = df["label"]

#     return X, y, df["text"]

# def balance_dataset(X, y, texts, method="smote"):
#     """
#     Balances dataset using different resampling techniques.
#     """
#     print(f"Applying {method}...")

#     sampler = None
#     if method == "oversample":
#         sampler = RandomOverSampler(sampling_strategy="auto")
#     elif method == "undersample":
#         sampler = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
#     elif method == "smote":
#         sampler = SMOTE(sampling_strategy="auto", random_state=42)
#     elif method == "smoteenn":
#         sampler = SMOTEENN(sampling_strategy="auto", random_state=42, n_jobs=-1)
#     elif method == "smotetomek":
#         sampler = SMOTETomek(sampling_strategy="auto", random_state=42, n_jobs=-1)
#     elif method == "adasyn":
#         sampler = ADASYN(sampling_strategy="minority", random_state=42, n_neighbors=5)
#     elif method == "nearmiss":
#         sampler = NearMiss(version=3, n_neighbors=3)
#     elif method == "tomek":
#         sampler = TomekLinks()
#     else:
#         raise ValueError("Invalid method. Choose a valid resampling technique.")

#     if sampler:
#         tqdm_bar = tqdm(total=len(X), desc=f"{method} in progress")
#         X_resampled, y_resampled = sampler.fit_resample(X, y)
#         tqdm_bar.update(len(X))  # Update progress after resampling is done
#         tqdm_bar.close()

#     df_balanced = pd.concat([pd.DataFrame(X_resampled, columns=X.columns), pd.DataFrame({"label": y_resampled})], axis=1)
#     df_balanced["text"] = np.random.choice(texts.values, len(df_balanced), replace=True)
#     return df_balanced

# def process_and_balance(df, method="smote", max_features=1000, pca_components=300):
#     X, y, texts = preprocess_features(df, max_features, pca_components)
#     return balance_dataset(X, y, texts, method)

# print("######### Training Data ##########")
# train_df = process_and_balance(train_df, method="smoteenn", max_features=500, pca_components=100)
# # print("######### Evaluation Data ##########")
# # eval_df = process_and_balance(eval_df, method="smoteenn", max_features=1000, pca_components=100)
# # print("######### Test Data ##########")
# # test_df = process_and_balance(test_df, method="smoteenn", max_features=1000, pca_components=100)

# # Check class distribution
# print(train_df["label"].value_counts())
# print(eval_df["label"].value_counts())
# print(test_df["label"].value_counts())

# def compute_text_correlation(df):
#     """Computes correlation of text between two classes using PCA-reduced BERT embeddings."""
#     class_0 = df[df["label"] == 0][[f"bert_pca_{i}" for i in range(100)]].mean().values
#     class_1 = df[df["label"] == 1][[f"bert_pca_{i}" for i in range(100)]].mean().values
#     correlation = 1 - cosine(class_0, class_1)
#     print(f"Text correlation between class 0 and class 1: {correlation:.4f}")

# compute_text_correlation(train_df)

In [ ]:
# # Define save path
# save_path = "/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/Generated_Datasets/4_Binary_Final"

# # Save processed datasets
# train_df.to_csv(f"{save_path}/smoteenn_train_final.csv", index=False)
# eval_df.to_csv(f"{save_path}/smoteenn_eval_final.csv", index=False)
# test_df.to_csv(f"{save_path}/smoteenn_test_final.csv", index=False)

# print("Datasets saved successfully!")

Datasets saved successfully!


In [ ]:
# read_path = "/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/Generated_Datasets/4_Binary_Final"

# # Save processed datasets
# train_df = pd.read_csv(f"{read_path}/smoteenn_train_final.csv")
# eval_df = pd.read_csv(f"{read_path}/smoteenn_eval_final.csv")
# test_df = pd.read_csv(f"{read_path}/smoteenn_test_final.csv")

# train_df.head(5)

## Model Configuration

In [8]:
import torch
import torch.nn as nn
import numpy as np
import random
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    RobertaConfig,
)

# Set global seed for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# Model name
MODEL_NAME = "roberta-base"

# Load configuration with dropout settings
config = RobertaConfig.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    hidden_dropout_prob=0.1,           # BERT uses 'hidden_dropout_prob'
    attention_probs_dropout_prob=0.1   # BERT uses 'attention_probs_dropout_prob'
)

# Optional: Enable gradient checkpointing for memory efficiency
config.gradient_checkpointing = True

# Load model with updated configuration
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    config=config
)

# Optional: Freeze base model if dataset is small
# Comment this section out if you want to fine-tune the entire model
for param in model.base_model.parameters():
    param.requires_grad = False

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Ensure padding/truncation setup
tokenizer.padding_side = "right"
tokenizer.truncation_side = "right"

# Confirm setup
print(f"✅ {MODEL_NAME} model and tokenizer loaded.")
print("🔍 Model architecture summary:")
print(model)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

✅ roberta-base model and tokenizer loaded.
🔍 Model architecture summary:
RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
            

### Tokenization

In [9]:
import torch
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np
from torch.utils.data import Dataset

# ✅ Sentence-pair Dataset Class for Hugging Face Transformers
class LazyCustomDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=512):
        self.sentence1 = df["text1"].tolist()
        self.sentence2 = df["text2"].tolist()
        self.labels = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

        print(f"✅ Dataset Size: {len(df)}")  # For confirmation

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        sent1 = self.sentence1[idx]
        sent2 = self.sentence2[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            sent1,
            sent2,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "token_type_ids": encoding.get("token_type_ids", torch.zeros_like(encoding["attention_mask"])).squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }

# ✅ Instantiate datasets
train_dataset = LazyCustomDataset(train_df, tokenizer)
val_dataset = LazyCustomDataset(eval_df, tokenizer)
test_dataset = LazyCustomDataset(test_df, tokenizer)


✅ Dataset Size: 74286
✅ Dataset Size: 16799
✅ Dataset Size: 16692


### Training Arguments


In [10]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=f"/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/3_Author_Change_Detection/Results/{MODEL_NAME}-results",

    # Evaluation
    eval_strategy="steps",
    eval_steps=600,
    save_strategy="steps",
    save_steps=600,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    # Training
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    weight_decay=0.01,
    max_grad_norm=1.0,
    lr_scheduler_type="cosine",
    warmup_steps=200,

    # Logging
    logging_dir=f"/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/3_Author_Change_Detection/Results/{MODEL_NAME}-logs",
    logging_strategy="steps",
    logging_steps=50,

    # System
    fp16=True,  # Enable if using a GPU that supports it
    dataloader_num_workers=4,  # Reduced for Colab/file-backed systems to avoid spawn issues

    # Misc
    report_to="none",
    run_name=f"{MODEL_NAME}-author-change-detection"
)


### Training

In [11]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.utils.class_weight import compute_class_weight
from transformers import Trainer

# Compute class weights from training labels
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df["label"].values
)

# Convert to torch tensor and move to appropriate device
class_weights = torch.tensor(class_weights, dtype=torch.float)
device = "cuda" if torch.cuda.is_available() else "cpu"
class_weights = class_weights.to(device)

# ✅ Compatible with latest HuggingFace Trainer
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss


In [12]:
import os

last_checkpoint = None

if os.path.exists(training_args.output_dir) and os.path.isdir(training_args.output_dir):
    checkpoints = [ckpt for ckpt in os.listdir(training_args.output_dir) if ckpt.startswith("checkpoint-")]

    if checkpoints:
        # Sort by checkpoint step number
        try:
            checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
            last_checkpoint = os.path.join(training_args.output_dir, checkpoints[-1])
        except ValueError:
            print("⚠️ Warning: Skipping checkpoint sorting due to unexpected format.")
            last_checkpoint = os.path.join(training_args.output_dir, checkpoints[-1])

print(f"📌 Last checkpoint found: {last_checkpoint}")


📌 Last checkpoint found: None


In [13]:
from transformers import Trainer, DataCollatorWithPadding, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
import numpy as np
from collections import Counter

# ✅ Evaluation Function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)

    conf_matrix = confusion_matrix(labels, preds)
    pred_counts = Counter(preds)

    print(f"\n📈 Prediction Distribution: {dict(pred_counts)}")
    print(f"📊 Confusion Matrix:\n{conf_matrix}")
    print("🔍 Per-Class Report:\n", classification_report(labels, preds, digits=4, zero_division=0))

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

# ✅ Trainer Setup
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# ✅ Start Training
if last_checkpoint:
    print(f"🔁 Resuming training from checkpoint: {last_checkpoint}")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("🚀 Starting training from scratch!")
    trainer.train()


🚀 Starting training from scratch!


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
600,0.675300,0.673938,0.696887,0.718516,0.696887,0.702711
1200,0.660400,0.658386,0.694029,0.724979,0.694029,0.700739
1800,0.664800,0.649028,0.690815,0.724802,0.690815,0.697747
2400,0.659500,0.643170,0.691886,0.725425,0.691886,0.698776



📈 Prediction Distribution: {np.int64(0): 7313, np.int64(1): 9486}
📊 Confusion Matrix:
[[4050 1829]
 [3263 7657]]
🔍 Per-Class Report:
               precision    recall  f1-score   support

           0     0.5538    0.6889    0.6140      5879
           1     0.8072    0.7012    0.7505     10920

    accuracy                         0.6969     16799
   macro avg     0.6805    0.6950    0.6822     16799
weighted avg     0.7185    0.6969    0.7027     16799


📈 Prediction Distribution: {np.int64(0): 7783, np.int64(1): 9016}
📊 Confusion Matrix:
[[4261 1618]
 [3522 7398]]
🔍 Per-Class Report:
               precision    recall  f1-score   support

           0     0.5475    0.7248    0.6238      5879
           1     0.8205    0.6775    0.7422     10920

    accuracy                         0.6940     16799
   macro avg     0.6840    0.7011    0.6830     16799
weighted avg     0.7250    0.6940    0.7007     16799


📈 Prediction Distribution: {np.int64(0): 7923, np.int64(1): 8876}
📊 Confusi

In [14]:
best_checkpoint_path = trainer.state.best_model_checkpoint
print(f"✅ Best checkpoint path based on F1 score: {best_checkpoint_path}")

✅ Best checkpoint path based on F1 score: /content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/3_Author_Change_Detection/Results/roberta-base-results/checkpoint-600


### Evaluation

In [15]:
predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(-1)
pred_probs = torch.nn.functional.softmax(torch.tensor(predictions.predictions), dim=-1).numpy()


📈 Prediction Distribution: {np.int64(0): 7135, np.int64(1): 9557}
📊 Confusion Matrix:
[[3925 1884]
 [3210 7673]]
🔍 Per-Class Report:
               precision    recall  f1-score   support

           0     0.5501    0.6757    0.6065      5809
           1     0.8029    0.7050    0.7508     10883

    accuracy                         0.6948     16692
   macro avg     0.6765    0.6904    0.6786     16692
weighted avg     0.7149    0.6948    0.7006     16692



### Store Results

In [16]:
import pandas as pd
from openpyxl import load_workbook
import os
from datetime import datetime
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# --------------------------------------
# 📁 Paths and Constants
# --------------------------------------
model_folder = "/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/3_Author_Change_Detection/Results"
results_file_path = os.path.join(model_folder, "Results.xlsx")
metrics_file_path = os.path.join(model_folder, "Evaluation_Metrics.xlsx")

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# --------------------------------------
# 📝 Save Per-Sample Predictions
# --------------------------------------
# Load existing results (if any)
try:
    existing_df = pd.read_excel(results_file_path)
except FileNotFoundError:
    existing_df = pd.DataFrame()

# Ensure alignment with test_df length
assert len(test_df) == len(preds) == len(pred_probs), "Mismatch in prediction lengths"

# Create new prediction DataFrame
df_results = pd.DataFrame({
    f"{MODEL_NAME}-Actual": test_df["label"].values,
    f"{MODEL_NAME}-Predicted": preds,
    f"{MODEL_NAME}-Prob_Positive": pred_probs[:, 1],
    f"{MODEL_NAME}-Prob_Negative": pred_probs[:, 0],
})

# Merge or overwrite matching columns in existing dataframe
for col in df_results.columns:
    existing_df[col] = df_results[col]

# Save to Excel
with pd.ExcelWriter(results_file_path, engine="openpyxl", mode="w") as writer:
    existing_df.to_excel(writer, index=False)

print(f"✅ Predictions saved to: {results_file_path}")

# --------------------------------------
# 📊 Evaluation Metrics
# --------------------------------------
# Extract ground truth
true_labels = [example["labels"] for example in test_dataset]

# Compute metrics
accuracy = accuracy_score(true_labels, preds)
precision, recall, f1, _ = precision_recall_fscore_support(true_labels, preds, average="weighted")
conf_matrix = confusion_matrix(true_labels, preds)

print(f"\n📊 Evaluation Metrics for {MODEL_NAME}:")
print(f"✅ Accuracy:  {accuracy:.4f}")
print(f"✅ Precision: {precision:.4f}")
print(f"✅ Recall:    {recall:.4f}")
print(f"✅ F1 Score:  {f1:.4f}")
print(f"\n✅ Confusion Matrix:\n{pd.DataFrame(conf_matrix)}")

# Build metrics record
metrics_data = {
    "Timestamp": timestamp,
    "Model": MODEL_NAME,
    "Accuracy": round(accuracy, 4),
    "Precision": round(precision, 4),
    "Recall": round(recall, 4),
    "F1 Score": round(f1, 4),
}

new_row_df = pd.DataFrame([metrics_data])

# Append to existing or create new
if os.path.exists(metrics_file_path):
    df_metrics = pd.read_excel(metrics_file_path)
    df_metrics = pd.concat([df_metrics, new_row_df], ignore_index=True)
else:
    df_metrics = new_row_df

# Save metrics
with pd.ExcelWriter(metrics_file_path, engine="openpyxl", mode="w") as writer:
    df_metrics.to_excel(writer, sheet_name="Metrics", index=False)

print(f"📂 Evaluation metrics saved to: {metrics_file_path}")

✅ Predictions saved to: /content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/3_Author_Change_Detection/Results/Results.xlsx

📊 Evaluation Metrics for roberta-base:
✅ Accuracy:  0.6948
✅ Precision: 0.7149
✅ Recall:    0.6948
✅ F1 Score:  0.7006

✅ Confusion Matrix:
      0     1
0  3925  1884
1  3210  7673
📂 Evaluation metrics saved to: /content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/3_Author_Change_Detection/Results/Evaluation_Metrics.xlsx


In [17]:
df_metrics

,Timestamp,Model,Accuracy,Precision,Recall,F1 Score
0,2025-07-20 19:40:18,albert-base-v2,0.6818,0.6964,0.6818,0.6867
1,2025-07-20 20:53:02,bert-base-uncased,0.6400,0.6908,0.6400,0.6484
2,2025-07-22 21:21:47,distilbert-base-uncased,0.6973,0.7147,0.6973,0.7026
3,2025-07-22 22:00:01,roberta-base,0.6948,0.7149,0.6948,0.7006


### Save Model

In [18]:
from transformers import AutoTokenizer

path = '/content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/3_Author_Change_Detection/Results'

# ✅ Define export path
export_path = os.path.join(path, f"{MODEL_NAME}_Model")

# ✅ Create the directory if it doesn't exist (no error if it does)
os.makedirs(export_path, exist_ok=True)

# ✅ Overwrite the model and tokenizer files
trainer.model.save_pretrained(export_path)
tokenizer.save_pretrained(export_path)

print(f"✅ Model and tokenizer successfully saved at: {export_path}")

✅ Model and tokenizer successfully saved at: /content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/3_Author_Change_Detection/Results/roberta-base_Model


### Retain the Best Checkpoint and Delete Others

In [19]:
import os
import shutil

# Get the parent directory where all checkpoints are saved
checkpoints_root = os.path.dirname(best_checkpoint_path)

# Loop through all items in the checkpoint directory
for subdir in os.listdir(checkpoints_root):
    full_path = os.path.join(checkpoints_root, subdir)

    # Remove everything except the best checkpoint
    if os.path.isdir(full_path) and full_path != best_checkpoint_path and "checkpoint" in subdir:
        print(f"🗑️ Removing checkpoint: {full_path}")
        shutil.rmtree(full_path)

print("✅ Only best checkpoint retained!")

🗑️ Removing checkpoint: /content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/3_Author_Change_Detection/Results/roberta-base-results/checkpoint-1800
🗑️ Removing checkpoint: /content/drive/MyDrive/PAN_LLM_Data_Generation_Normal_Prompt_Gemini/3_Author_Change_Detection/Results/roberta-base-results/checkpoint-2400
✅ Only best checkpoint retained!


### Load Model Again

In [ ]:
# from transformers import AutoModelForSequenceClassification, AutoTokenizer

# model = AutoModelForSequenceClassification.from_pretrained(export_path)
# tokenizer = AutoTokenizer.from_pretrained(export_path)